In [29]:
import pandas as pd


calorim = pd.read_csv("../../results/cooling/23/sveGodine.csv") 
zones = pd.read_csv("../../results/cooling/23/zones.csv") 
fcu_files = pd.read_csv("../../results/23/fcu_type_id2_with_zone.csv") 
koef = pd.read_csv("../../data/fcu_hydraulic_model.csv")


calorim['timestamp'] = pd.to_datetime(calorim['timestamp']).dt.round('min')
zones['timestamp'] = pd.to_datetime(zones['timestamp']).dt.round('min')
fcu_files['timestamp'] = pd.to_datetime(fcu_files['timestamp']).dt.floor('min')
koef['timestamp'] = pd.to_datetime(koef['timestamp']).dt.round('min')

calorim = calorim.sort_values('timestamp')
zones = zones.sort_values('timestamp')

merged = pd.merge_asof(zones, calorim, on='timestamp', direction='nearest')
print(len(merged))

final_merged1 = pd.merge(merged, fcu_files, on=['timestamp', 'zone_id'], how='inner')
print(len(final_merged1))

koef = koef.rename(columns={'timestamp': 'timestamp_koef'})

final_merged = pd.merge(final_merged1, koef, on='fcu_id', how='left')
print(len(final_merged))

final_merged = final_merged.sort_values('timestamp')
final_merged.to_csv("../../results/cooling/23/merged_data.csv", index=False)

print(len(final_merged))
print(final_merged.head())


30961
52127
52127
52127
   zone_id           batch_timestamp_x           timestamp  zone_temperature  \
0      210  2018-03-23 03:27:01.575521 2018-03-23 03:27:00             18.52   
1      209  2018-03-23 03:27:01.575521 2018-03-23 03:27:00             19.39   
2      209  2018-03-23 03:27:01.575521 2018-03-23 03:27:00             19.39   
3      208  2018-03-23 03:27:01.575521 2018-03-23 03:27:00             18.24   
4      208  2018-03-23 03:27:01.575521 2018-03-23 03:27:00             18.24   

   zone_fan_speed  zone_valve_duty_cycle  smart_control  local_switch  \
0             0.0                    NaN          False           0.0   
1             0.0                    NaN          False           0.0   
2             0.0                    NaN          False           0.0   
3             0.0                    NaN          False           0.0   
4             0.0                    NaN          False           0.0   

   temperature_setpoint  calorimeter_id  ... in_cooling_

In [32]:


CP_WATER = 4184  

scale_factor = 0.1  
final_merged['mass_flow_kg_s'] = (final_merged['mass_volume_flow'] * final_merged['flow_share']  / 100 ) / 3600

final_merged['delta_T'] = final_merged['supply_temperature']  - final_merged['return_medium_temperature']

final_merged['calculated_thermal_power'] = (
    final_merged['mass_flow_kg_s'] * CP_WATER * final_merged['delta_T'] / 1000
)

print(final_merged[['thermal_power', 'calculated_thermal_power', 'mass_flow_kg_s', 'delta_T']].head(20))


final_merged.to_csv("../../results/cooling/23/calc_thermal.csv", index=False)


    thermal_power  calculated_thermal_power  mass_flow_kg_s  delta_T
0             0.3                  0.228734       -0.004157   -13.15
1             0.3                  0.221869       -0.003902   -13.59
2             0.3                  0.232053       -0.004063   -13.65
3             0.3                  0.185928       -0.003445   -12.90
4             0.3                  0.202519       -0.003645   -13.28
5             0.3                  0.243822       -0.004288   -13.59
6             0.3                  0.223502       -0.003913   -13.65
7             0.3                  0.217327       -0.003822   -13.59
8             0.3                  0.267971       -0.004546   -14.09
9             0.3                  0.275612       -0.004613   -14.28
10            0.3                  0.198166       -0.003470   -13.65
11            0.3                  0.186597       -0.003358   -13.28
19            0.3                  0.234502       -0.004613   -12.15
18            0.3                 